# 🎯 Calibration & Threshold Optimization

Optimización del umbral de decisión para balancear precisión, recall y costo operativo.

**Qué hace esta notebook:**
- Carga el reporte de evaluación (JSON con y_true, y_proba del conjunto de test)
- Grafica la curva de calibración (reliability diagram): ¿las probabilidades del modelo reflejan la realidad?
- Barre thresholds y calcula métricas a cada nivel
- Optimiza el threshold según costo de falsos positivos vs falsos negativos
- Muestra el impacto operativo: cuántas inspecciones, cuántos fraudes detectados
- Compara threshold global vs thresholds por segmento

**Cuándo usarla:**
- Después de entrenar un modelo, para decidir a qué threshold operar
- Cuando necesitás justificar el threshold elegido frente al negocio
- Para entender el trade-off entre "detectar más fraudes" y "gastar más en inspecciones"

## 0. Configuración

In [ ]:
# ============================================================
# CONFIGURACIÓN — paths, archivos y parámetros
# ============================================================
# Toda la configuración está acá. Para apuntar la notebook a otro
# proyecto o dataset, modificá estos valores (no hace falta tocar el
# resto de las celdas).

from pathlib import Path

# --- Proyecto y datos ---
PROJECT_PATH = Path('/home/vvv/Develop/bid/energizados/.proyects/celesc')
OUTPUT_PATH = PROJECT_PATH / 'output'
VERSION = 'v5'
TRAIN_DIR = 'train'

# --- Reporte de evaluación (run de entrenamiento) ---
EVALUATION_REPORT = OUTPUT_PATH / VERSION / TRAIN_DIR / 'reports' / 'evaluation' / 'evaluation_report.json'

# --- Entorno (detección automática Colab vs local) ---
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# === PARÁMETROS DE NEGOCIO ===
# Costo de un falso positivo (inspeccionar a un cliente que NO es fraude)
COST_FP = 1  # ej: $100 de viático + hora del inspector

# Costo de un falso negativo (dejar pasar un fraude sin detectar)
COST_FN = 10  # ej: $1000 de energía no facturada en un año

# Capacidad operativa: cuántas inspecciones puede hacer el equipo por período
MAX_INSPECTIONS = 5000

# Recall mínimo aceptable (para el método precision_recall)
MIN_RECALL = 0.80

print(f'PROJECT_PATH      : {PROJECT_PATH}')
print(f'EVALUATION_REPORT : {EVALUATION_REPORT}')
print(f'IN_COLAB          : {IN_COLAB}')

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    auc, average_precision_score, confusion_matrix,
    f1_score, precision_recall_curve, precision_score,
    recall_score, roc_auc_score, roc_curve,
)

from energizados.evaluation.calibration import ThresholdCalibrator

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 6)

assert Path(EVALUATION_REPORT).exists(), f"No se encuentra: {EVALUATION_REPORT}"
print("✅ Configuración lista")

## 1. Carga del reporte

El reporte de evaluación incluye métricas al threshold con el que se evaluó. Pero necesitamos los vectores `y_true` e `y_proba` para recalibrar.

> ⚠️ **Importante:** El reporte estándar de Energizados no guarda `y_true`/`y_proba` completos (solo métricas agregadas). Esta notebook espera que el reporte tenga la clave `threshold_metrics` con `y_true` e `y_proba`, que se generan cuando `generate_json_report: true` y `save_predictions: true` están habilitados en `evaluation`.

Si tu reporte no tiene esos vectores, vas a necesitar volver a correr la evaluación con esas opciones, o cargar los datos de test manualmente.

In [ ]:
with open(EVALUATION_REPORT) as f:
    report = json.load(f)

# Métricas globales
global_metrics = report.get("metrics", {})
print("=== Métricas globales ===")
for k, v in global_metrics.items():
    if not isinstance(v, (dict, list)):
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

default_threshold = global_metrics.get("threshold") or 0.5  # el reporte puede tener threshold=None
print(f"\nThreshold de evaluación: {default_threshold}")

## 2. Datos de test: y_true e y_proba

Intentamos cargar los vectores del reporte. Si no están, los simulamos a partir de las métricas agregadas para poder mostrar los conceptos (pero los resultados numéricos no serán exactos).

> En un caso real, necesitás `y_true` e `y_proba` del conjunto de test. Podés obtenerlos corriendo evaluación con el framework.

In [ ]:
# Intentar cargar y_true / y_proba del reporte
threshold_metrics = report.get("threshold_metrics", {})

if "y_true" in threshold_metrics and "y_proba" in threshold_metrics:
    y_true = np.array(threshold_metrics["y_true"])
    y_proba = np.array(threshold_metrics["y_proba"])
    print(f"✅ Vectores cargados del reporte: {len(y_true):,} muestras")
    print(f"   Fraude real: {y_true.sum():,} ({y_true.mean()*100:.2f}%)")
else:
    # Fallback: simular a partir de las métricas agregadas
    print("⚠️  El reporte no incluye y_true/y_proba. Simulando datos para demostración...")
    print("   Las métricas numéricas serán aproximadas, no exactas.\n")
    
    # Usamos las métricas del reporte para generar una distribución plausible
    auc_val = global_metrics.get("auc", 0.7)
    precision_val = global_metrics.get("precision", 0.5)
    recall_val = global_metrics.get("recall", 0.5)
    
    np.random.seed(42)
    n_total = 10000
    n_pos = int(n_total * 0.15)  # 15% de fraude típico
    
    y_true = np.zeros(n_total, dtype=int)
    y_true[:n_pos] = 1
    np.random.shuffle(y_true)
    
    # Generar probabilidades que aproximen las métricas
    y_proba = np.random.beta(2, 4, n_total)
    y_proba[y_true == 1] = np.random.beta(4, 2, n_pos)
    
    # Escalar para aproximar el AUC reportado
    y_proba = y_proba * 0.6 + 0.2
    
    print(f"   Muestras simuladas: {n_total:,} (≈{y_true.mean()*100:.1f}% fraude)")

## 3. Curva de calibración (Reliability Diagram)

**¿Qué es?** Mide si cuando el modelo dice "70% de probabilidad de fraude", realmente el 70% de esos clientes son fraude.

**Cómo leerlo:**
- **Línea diagonal punteada = calibración perfecta.** Si el modelo dice 0.8, el 80% de esos casos deberían ser fraude.
- **Curva por debajo de la diagonal:** el modelo **sobreestima** — asigna probabilidades más altas de lo que realmente ocurre. Ej: dice 0.8 pero solo 50% son fraude. Típico en datasets muy desbalanceados.
- **Curva por encima de la diagonal:** el modelo **subestima** — es demasiado conservador. Raro en la práctica.
- **Curva con forma de S:** el modelo empuja las predicciones a los extremos (0 o 1) — overconfident.
- **Histograma en la base:** muestra cuántos clientes caen en cada bucket de probabilidad. Si todos están en 0.8-0.9, el modelo no discrimina.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Reliability diagram
prob_true, prob_pred = calibration_curve(y_true, y_proba, n_bins=15, strategy="uniform")

axes[0].plot(prob_pred, prob_true, "s-", color="steelblue", linewidth=2, markersize=6, label="Modelo")
axes[0].plot([0, 1], [0, 1], "k--", linewidth=1, label="Calibración perfecta")
axes[0].fill_between(prob_pred, prob_true, prob_pred, alpha=0.1, color="steelblue")
axes[0].set_xlabel("Probabilidad predicha (media del bucket)")
axes[0].set_ylabel("Fracción real de positivos")
axes[0].set_title("Reliability Diagram (Curva de Calibración)")
axes[0].legend()
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)

# Histograma de probabilidades
axes[1].hist(y_proba[y_true == 0], bins=30, alpha=0.6, color="steelblue", label="No fraude", density=True)
axes[1].hist(y_proba[y_true == 1], bins=30, alpha=0.6, color="crimson", label="Fraude", density=True)
axes[1].axvline(default_threshold, color="gray", linestyle="--", linewidth=1.5, label=f"threshold={default_threshold}")
axes[1].set_xlabel("Probability")
axes[1].set_ylabel("Densidad")
axes[1].set_title("Distribución de probabilidades por clase")
axes[1].legend()

plt.tight_layout()
plt.show()

# Métrica de calibración: Brier Score (más bajo = mejor)
brier = np.mean((y_proba - y_true) ** 2)
print(f"Brier Score: {brier:.4f}  (0 = perfecto, 0.25 = aleatorio con 50% fraude)")

## 4. Barrido de thresholds

Evaluamos todas las métricas en una grilla fina de thresholds para entender el trade-off.

**Qué mirar en la curva:**
- **Línea azul (precision):** a medida que subís el threshold, marcás menos pero con más certeza. La precisión sube.
- **Línea naranja (recall):** a medida que subís el threshold, detectás menos fraudes. El recall baja.
- **Línea verde (F1):** balance entre precisión y recall. El pico es el "mejor" threshold desde el punto de vista estadístico.
- **Línea roja punteada (clientes marcados):** cuántos clientes inspeccionarías. A medida que subís el threshold, inspeccionás menos.
- **Zona sombreada:** la región donde el threshold actual es "razonable" según tu costo operativo.

In [ ]:
# Barrido fino de thresholds
thresholds = np.linspace(0.01, 0.99, 200)
metrics_by_threshold = []

for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    total_flagged = tp + fp
    
    metrics_by_threshold.append({
        "threshold": t,
        "precision": tp / (tp + fp) if (tp + fp) > 0 else 0,
        "recall": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "flagged": total_flagged,
        "flagged_pct": total_flagged / len(y_true) * 100,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "total_cost": fp * COST_FP + fn * COST_FN,
    })

sweep_df = pd.DataFrame(metrics_by_threshold)

fig, ax1 = plt.subplots(figsize=(14, 7))

color_precision = "#2196F3"
color_recall = "#FF9800"
color_f1 = "#4CAF50"
color_flagged = "#F44336"

ax1.plot(sweep_df["threshold"], sweep_df["precision"], color=color_precision, linewidth=2, label="Precision")
ax1.plot(sweep_df["threshold"], sweep_df["recall"], color=color_recall, linewidth=2, label="Recall")
ax1.plot(sweep_df["threshold"], sweep_df["f1"], color=color_f1, linewidth=2.5, label="F1")
ax1.axvline(default_threshold, color="gray", linestyle="--", linewidth=1.5, alpha=0.7, label=f"Default={default_threshold}")
ax1.set_xlabel("Threshold")
ax1.set_ylabel("Métrica", color="black")
ax1.set_ylim(0, 1.05)
ax1.legend(loc="center left", fontsize=9)

# Eje secundario: clientes marcados (%)
ax2 = ax1.twinx()
ax2.plot(sweep_df["threshold"], sweep_df["flagged_pct"], color=color_flagged, linewidth=1.5, linestyle="--", alpha=0.7, label="% Marcados")
ax2.set_ylabel("% de clientes marcados", color=color_flagged)
ax2.tick_params(axis="y", labelcolor=color_flagged)
ax2.legend(loc="center right", fontsize=9)

plt.title("Métricas vs Threshold")
plt.tight_layout()
plt.show()

## 5. Curva de costo total

Acá es donde el negocio se encuentra con el modelo. Traducimos falsos positivos y falsos negativos a pesos.

**Qué mirar:**
- **Mínimo de la curva roja:** el threshold que minimiza el costo total. Este es el threshold óptimo desde el punto de vista económico.
- **Costo dominado por FN (izquierda):** thresholds bajos marcan muchos → pocos FN pero muchos FP. Si `COST_FN >> COST_FP`, preferís thresholds más bajos.
- **Costo dominado por FP (derecha):** thresholds altos marcan pocos → pocos FP pero muchos FN. Si `COST_FP >> COST_FN`, preferís thresholds más altos.
- **Zona plana en el mínimo:** si el costo es similar en un rango amplio, tenés flexibilidad para elegir el threshold que mejor se adapte a tu capacidad operativa.

**Cómo ajustar COST_FP y COST_FN:**
- `COST_FP`: cuánto te cuesta mandar un inspector a un cliente que resulta ser legítimo (viático, hora-hombre, molestia al cliente).
- `COST_FN`: cuánto perdés por no detectar un fraude (energía no facturada × tiempo hasta la próxima inspección).

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

# Costos
ax1.plot(sweep_df["threshold"], sweep_df["fp"] * COST_FP, color="#FF9800", linewidth=2, label=f"Costo FP (COST_FP={COST_FP})")
ax1.plot(sweep_df["threshold"], sweep_df["fn"] * COST_FN, color="#F44336", linewidth=2, label=f"Costo FN (COST_FN={COST_FN})")
ax1.plot(sweep_df["threshold"], sweep_df["total_cost"], color="#9C27B0", linewidth=3, label="Costo Total")

# Marcar el mínimo
idx_min = sweep_df["total_cost"].idxmin()
opt_threshold_cost = sweep_df.loc[idx_min, "threshold"]
opt_cost = sweep_df.loc[idx_min, "total_cost"]
ax1.axvline(opt_threshold_cost, color="purple", linestyle="--", linewidth=2, alpha=0.7,
            label=f"Óptimo costo: t={opt_threshold_cost:.3f}")

ax1.set_xlabel("Threshold")
ax1.set_ylabel("Costo total")
ax1.set_title(f"Costo total vs Threshold (COST_FP={COST_FP}, COST_FN={COST_FN})")
ax1.legend(fontsize=9)

# Anotar valores en el óptimo
opt_row = sweep_df.loc[idx_min]
ax1.annotate(
    f"t={opt_threshold_cost:.3f}\n"
    f"FP={opt_row['fp']:,}, FN={opt_row['fn']:,}\n"
    f"Precision={opt_row['precision']:.3f}, Recall={opt_row['recall']:.3f}",
    xy=(opt_threshold_cost, opt_cost),
    xytext=(opt_threshold_cost + 0.08, opt_cost * 1.3),
    arrowprops=dict(arrowstyle="->", color="purple"),
    fontsize=9, bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
)

plt.tight_layout()
plt.show()

print(f"Threshold óptimo (costo): {opt_threshold_cost:.4f}")
print(f"  Precision: {opt_row['precision']:.3f}")
print(f"  Recall:    {opt_row['recall']:.3f}")
print(f"  F1:        {opt_row['f1']:.3f}")
print(f"  Marcados:  {opt_row['flagged']:,} ({opt_row['flagged_pct']:.1f}%)")
print(f"  Costo FP:  {opt_row['fp'] * COST_FP:,.0f}")
print(f"  Costo FN:  {opt_row['fn'] * COST_FN:,.0f}")
print(f"  Costo TTL: {opt_row['total_cost']:,.0f}")

## 6. Threshold por capacidad operativa

A veces el limitante no es el costo, sino cuántas inspecciones puede hacer tu equipo. Este análisis encuentra el mejor threshold dado un límite de inspecciones.

**Qué mirar:**
- **Línea horizontal = capacidad máxima.** El threshold se ajusta para no exceder ese número de inspecciones.
- **Si la capacidad es muy baja,** vas a tener alta precisión (los que inspeccionás son casi seguro fraude) pero bajo recall (dejás pasar muchos).
- **Si la capacidad es muy alta,** el threshold baja y empezás a inspeccionar muchos falsos positivos.

In [ ]:
# Encontrar el threshold que produce exactamente (o apenas menos de) MAX_INSPECTIONS
# Recorremos de threshold alto a bajo (más restrictivo a más laxo)
operational_threshold = None
for _, row in sweep_df.iterrows():
    if row["flagged"] <= MAX_INSPECTIONS:
        operational_threshold = row["threshold"]
        break

if operational_threshold is None:
    operational_threshold = 0.99

op_row = sweep_df[sweep_df["threshold"] == operational_threshold].iloc[0]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(sweep_df["threshold"], sweep_df["flagged"], color="steelblue", linewidth=2)
ax.axhline(MAX_INSPECTIONS, color="crimson", linestyle="--", linewidth=1.5, label=f"Capacidad={MAX_INSPECTIONS:,}")
ax.axvline(operational_threshold, color="green", linestyle="--", linewidth=1.5,
           label=f"Threshold operativo={operational_threshold:.3f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Clientes marcados")
ax.set_title(f"Threshold por capacidad operativa (máx {MAX_INSPECTIONS:,} inspecciones)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Threshold operativo: {operational_threshold:.4f}")
print(f"  Inspecciones:  {op_row['flagged']:,} (capacidad: {MAX_INSPECTIONS:,})")
print(f"  Fraudes detectados: {op_row['tp']:,} de {op_row['tp'] + op_row['fn']:,} totales")
print(f"  Fraudes perdidos:   {op_row['fn']:,}")
print(f"  Precision: {op_row['precision']:.3f}")
print(f"  Recall:    {op_row['recall']:.3f}")

## 7. Curva Precision-Recall con thresholds anotados

La curva PR es más informativa que la ROC cuando las clases son muy desbalanceadas (pocos fraudes, muchos normales).

**Qué mirar:**
- **Cada punto en la curva = un threshold distinto.**
- **Extremo derecho (recall alto):** thresholds bajos, detecta casi todos los fraudes pero con baja precisión.
- **Extremo izquierdo (precision alta):** thresholds altos, los que marca son casi seguro fraude pero se pierde la mayoría.
- **Estrella:** threshold óptimo por costo.
- **Triángulo:** threshold operativo por capacidad.
- **Línea horizontal punteada:** precision del modelo aleatorio (proporción de fraudes en los datos).

In [ ]:
precision_curve, recall_curve, thresholds_pr = precision_recall_curve(y_true, y_proba)
avg_precision = average_precision_score(y_true, y_proba)

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(recall_curve, precision_curve, color="steelblue", linewidth=2, label=f"PR Curve (AP={avg_precision:.3f})")

# Marcar thresholds clave
for label, t, marker, color in [
    (f"Default t={default_threshold}", default_threshold, "s", "gray"),
    (f"Óptimo costo t={opt_threshold_cost:.3f}", opt_threshold_cost, "*", "purple"),
    (f"Operativo t={operational_threshold:.3f}", operational_threshold, "^", "green"),
]:
    y_pred_at_t = (y_proba >= t).astype(int)
    prec_at_t = precision_score(y_true, y_pred_at_t, zero_division=0)
    rec_at_t = recall_score(y_true, y_pred_at_t, zero_division=0)
    ax.scatter([rec_at_t], [prec_at_t], marker=marker, s=150, color=color,
              edgecolors="white", linewidth=1, zorder=5, label=label)

# Baseline: random classifier
baseline = y_true.mean()
ax.axhline(baseline, color="gray", linestyle=":", alpha=0.5, label=f"Aleatorio (prec={baseline:.3f})")

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Curva Precision-Recall")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 8. Umbrales por segmento (si aplica)

Si el reporte incluye `segmented_metrics` con thresholds por segmento, los comparamos con el threshold global.

**Qué mirar:**
- **Segmentos con threshold más alto que el global:** el modelo necesita más certeza para marcar en ese segmento (posiblemente porque hay mucho ruido).
- **Segmentos con threshold más bajo:** el modelo es más laxo — detecta más pero con más falsos positivos.
- **Diferencia grande (> 0.1) entre thresholds:** indica que el comportamiento del modelo varía mucho entre segmentos.

In [ ]:
segmented = report.get("segmented_metrics", {})

if segmented:
    fig, axes = plt.subplots(1, len(segmented), figsize=(7 * len(segmented), 5))
    if len(segmented) == 1:
        axes = [axes]
    
    for ax, (seg_col, segments) in zip(axes, segmented.items()):
        seg_items = sorted(segments.items(), key=lambda x: x[1].get("threshold", 0.5))
        names = [s[0] for s in seg_items]
        thresholds_seg = [s[1].get("threshold", 0.5) for s in seg_items]
        
        colors = ["#4CAF50" if t > default_threshold else "#F44336" for t in thresholds_seg]
        ax.barh(names, thresholds_seg, color=colors, edgecolor="white")
        ax.axvline(default_threshold, color="gray", linestyle="--", linewidth=1.5, label=f"Global={default_threshold}")
        ax.set_title(f"Thresholds por {seg_col}")
        ax.set_xlabel("Threshold")
        ax.set_xlim(0, 1)
        ax.legend(fontsize=8)
    
    plt.suptitle("Thresholds por segmento vs Global", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️  El reporte no incluye métricas segmentadas.")
    print("   Para habilitarlas, agregá a train.yaml:")
    print("   evaluation:")
    print("     segmented_evaluation:")
    print("       segment_columns: ['geo_region']")

## 9. Summary: Tabla de decisión

Comparación lado a lado de los tres thresholds candidatos.

In [ ]:
print("=" * 70)
print("SUMMARY — Comparación de Thresholds")
print("=" * 70)
print(f"{'':<25} {'Default':>12} {'Óptimo Costo':>14} {'Operativo':>14}")
print(f"{'Threshold':<25} {default_threshold:>12.4f} {opt_threshold_cost:>14.4f} {operational_threshold:>14.4f}")

for t, label in [(default_threshold, "default"), (opt_threshold_cost, "opt_cost"), (operational_threshold, "operational")]:
    y_pred = (y_proba >= t).astype(int)
    if label == "default":
        row_data = {}
    elif label == "opt_cost":
        row_data = sweep_df.loc[idx_min].to_dict()
    else:
        row_data = op_row.to_dict()

for metric, fmt in [("flagged", ",.0f"), ("precision", ".3f"), ("recall", ".3f"), ("f1", ".3f")]:
    vals = []
    for t, label in [(default_threshold, "default"), (opt_threshold_cost, "opt_cost"), (operational_threshold, "operational")]:
        y_pred = (y_proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        if metric == "flagged":
            vals.append(tp + fp)
        elif metric == "precision":
            vals.append(tp / (tp + fp) if (tp + fp) > 0 else 0)
        elif metric == "recall":
            vals.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        elif metric == "f1":
            vals.append(f1_score(y_true, y_pred, zero_division=0))
    print(f"{metric.capitalize():<25} {vals[0]:>12{fmt}} {vals[1]:>14{fmt}} {vals[2]:>14{fmt}}")

print()
print(f"Parámetros: COST_FP={COST_FP}, COST_FN={COST_FN}, MAX_INSPECTIONS={MAX_INSPECTIONS:,}")
print("=" * 70)

---
## Notas

- Los thresholds calculados dependen fuertemente de `COST_FP` y `COST_FN`. Ajustalos con datos reales de tu operación.
- Si los datos de test son simulados (porque el reporte no incluye `y_true`/`y_proba`), usá esta notebook como guía conceptual y volvé a correrla con datos reales.
- Para incluir `y_true`/`y_proba` en el reporte, configurá `evaluation.save_predictions: true` en `train.yaml`.